<a href="https://colab.research.google.com/github/jeffheaton/app_deep_learning/blob/main/t81_558_class_03_3_pandas_grouping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# T81-558: Applications of Deep Neural Networks

**Module 3: Working with Tabular Data**

* Instructor: [Jeff Heaton](https://sites.washu.edu/jeffheaton/), McKelvey School of Engineering, [Washington University in St. Louis](https://engineering.washu.edu/index.html)
* For more information visit the [class website](https://sites.washu.edu/jeffheaton/t81-558/).

# Module 3 Material

Main video lecture:

- Part 3.1 Should Neural Networks be Used for Tabular Data [[Video]]() [[Notebook]](t81_558_class_03_1_python_pandas.ipynb)
- Part 3.2 Encoding Categorical Values in Pandas [[Video]]() [[Notebook]](t81_558_class_03_2_pandas_cat.ipynb)
- **Part 3.3 Grouping, Sorting and Shuffling** [[Video]]() [[Notebook]](t81_558_class_03_3_pandas_grouping.ipynb)
- Part 3.4 Pandas Functional [[Video]]() [[Notebook]](t81_558_class_03_4_pandas_functional.ipynb)
- Part 3.5 Feature Engineering [[Video]]() [[Notebook]](t81_558_class_03_5_pandas_features.ipynb)

# Google Colab Instructions

The following code ensures that Google Colab is running and maps Google Drive if needed.

In [1]:
try:
    import google.colab
    COLAB = True
    print("Note: using Google Colab")
except ImportError:
    COLAB = False
    print("Note: not using Google Colab")

Note: using Google Colab


# Part 3.3: Grouping, Sorting, and Shuffling

We will take a look at a few ways to affect an entire Pandas data frame. These techniques will allow us to group, sort, and shuffle data sets. These are all essential operations for both data preprocessing and evaluation.

## Shuffling a Dataset

There may be information lurking in the order of the rows of your dataset. Unless you are dealing with time-series data, the order of the rows should not be significant. Consider whether your training set included employees in a company. Perhaps this dataset is ordered by the number of years each employee has been with the company. It is okay to include an individual column for years of service. However, having the data in this order might be problematic.

Consider what would happen if you split the data into training and validation sets. You could end up with your validation set containing only newer employees, and the training set containing only longer-term employees. Separating the data into k folds for k-fold cross-validation could lead to similar problems. Because of these issues, it is important to shuffle the data set.

Often, shuffling and reindexing are both performed together. Shuffling randomizes the order of the data set. However, it does not change the Pandas row numbers. The following code demonstrates a reshuffle. Notice that the program has not reset the row indexes' first column. Generally, this will not cause any issues and allows tracing the data back to the original order. However, I usually prefer to reset this index. I reason that I typically do not care about the initial position, and there are a few instances where this unordered index can cause issues.

In [2]:
import numpy as np
import pandas as pd

df = pd.read_csv(
    "https://data.heatonresearch.com/data/t81-558/auto-mpg.csv", na_values=["NA", "?"]
)

# np.random.seed(42) # Uncomment this line to get the same shuffle each time
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

pd.set_option("display.max_columns", 7)
pd.set_option("display.max_rows", 5)
display(df)

,mpg,cylinders,displacement,...,year,origin,name
0,33.0,4,91.0,...,76,3,honda civic
1,28.0,4,120.0,...,82,1,ford ranger
...,...,...,...,...,...,...,...
396,37.7,4,89.0,...,81,3,toyota tercel
397,26.0,4,97.0,...,73,2,volkswagen super beetle


### Sorting a Data Set

While it is always good to shuffle a data set before training, during training, and preprocessing, you may also wish to sort the data set. Sorting the data set lets you order the rows in ascending or descending order for one or more columns. The following code sorts the MPG dataset by name and displays the first car.

In [3]:
df = pd.read_csv(
    "https://data.heatonresearch.com/data/t81-558/auto-mpg.csv", na_values=["NA", "?"]
)

df = df.sort_values(by="name", ascending=True)
print(f"The first car is: {df['name'].iloc[0]}")

pd.set_option("display.max_columns", 7)
pd.set_option("display.max_rows", 5)
display(df)

The first car is: amc ambassador brougham


,mpg,cylinders,displacement,...,year,origin,name
96,13.0,8,360.0,...,73,1,amc ambassador brougham
9,15.0,8,390.0,...,70,1,amc ambassador dpl
...,...,...,...,...,...,...,...
325,44.3,4,90.0,...,80,2,vw rabbit c (diesel)
293,31.9,4,89.0,...,79,2,vw rabbit custom


### Grouping a Data Set

Grouping is a common operation on datasets. Structured Query Language (SQL) calls this operation a "GROUP BY." Programmers use grouping to summarize data. As a result, the summarization row count will usually shrink, and you cannot undo the grouping. Because of this loss of information, it is essential to keep your original data before the grouping.

We use the Auto MPG dataset to demonstrate grouping.


In [4]:
df = pd.read_csv(
    "https://data.heatonresearch.com/data/t81-558/auto-mpg.csv", na_values=["NA", "?"]
)

pd.set_option("display.max_columns", 7)
pd.set_option("display.max_rows", 5)
display(df)

,mpg,cylinders,displacement,...,year,origin,name
0,18.0,8,307.0,...,70,1,chevrolet chevelle malibu
1,15.0,8,350.0,...,70,1,buick skylark 320
...,...,...,...,...,...,...,...
396,28.0,4,120.0,...,82,1,ford ranger
397,31.0,4,119.0,...,82,1,chevy s-10


You can use the above dataset with the group to generate summaries. For example, the following code will group cylinders by the average (mean). This code will provide the grouping. In addition to **mean**, you can use other aggregating functions, such as **sum** or **count**.

In [5]:
g = df.groupby("cylinders")["mpg"].mean()
g

,mpg
cylinders,
3,20.550000
4,29.286765
5,27.366667
6,19.985714
8,14.963107


It might be useful to have these **mean** values as a dictionary.


In [6]:
d = g.to_dict()
d

{3: 20.55,
 4: 29.28676470588235,
 5: 27.366666666666664,
 6: 19.985714285714284,
 8: 14.963106796116506}

A dictionary allows you to access an individual element quickly. For example, you could quickly look up the mean for six-cylinder cars. You will see that target encoding, introduced later in this module, uses this technique.


In [7]:
d[6]

19.985714285714284

The code below shows how to count the number of rows that match each cylinder count.


In [8]:
df.groupby("cylinders")["mpg"].count().to_dict()

{3: 4, 4: 204, 5: 3, 6: 84, 8: 103}